# Concept Bottleneck Model — Chest X-ray Interpretability Pipeline

**How to use:**  
Run cells top-to-bottom once. Steps 1, 2, 3, 5, 6, 7, 8 recompute every run.  
Step 4 (image encoding) is the only cached step — if `img_emb_*.npy` files exist it loads them.  

To redo a step: re-run its cell. To force Step 4 to re-encode: delete `img_emb_*.npy` first.  

⚠ If you flip `USE_SUBSAMPLE`, delete all `img_emb_*.npy` files before rerunning.  
The alignment assert in Step 4 will catch stale cache and stop with a clear message.

In [1]:
# ── Uncomment to inspect folder structure ────────────────────────────────
# import os, re
#
# def explore(path, max_show=3, max_depth=4):
#     def walk(p, depth=0):
#         if depth > max_depth:
#             return
#         try:
#             entries = sorted(os.listdir(p))
#         except (PermissionError, NotADirectoryError):
#             return
#         entries = [e for e in entries if not e.startswith('.')]
#         dirs  = [e for e in entries if os.path.isdir(os.path.join(p, e))]
#         files = [e for e in entries if not os.path.isdir(os.path.join(p, e))]
#         ind = '  ' * depth
#         groups = {}
#         for d in dirs:
#             groups.setdefault(re.sub(r'\d+', '#', d), []).append(d)
#         for pattern, members in groups.items():
#             for d in members[:max_show]:
#                 print(f"{ind}{d}/")
#                 walk(os.path.join(p, d), depth + 1)
#             if len(members) > max_show:
#                 print(f"{ind}... {len(members) - max_show} more matching '{pattern}'")
#         for f in files[:max_show]:
#             print(f"{ind}{f}")
#         if len(files) > max_show:
#             print(f"{ind}... {len(files) - max_show} more files")
#     print(f"{os.path.basename(path.rstrip('/'))}/")
#     walk(path, 1)
#
# explore("CheXpert-v1.0-small")


## Config
All knobs live here. Nothing else defines these variables.

In [2]:
import pandas as pd
df = pd.read_csv("CheXpert-v1.0-small/CheXpert-v1.0-small/train.csv")
print(df["Path"].iloc[0])

CheXpert-v1.0-small/train/patient00001/study1/view1_frontal.jpg


In [3]:
from pathlib import Path

# ── Experiment ──────────────────────────────────────────────────────────
SEED          = 42
USE_SUBSAMPLE = True   # True → sample ~TARGET_ROWS rows; False → full dataset
TARGET_ROWS   = 10_000  # only read when USE_SUBSAMPLE = True
TARGET_LABEL  = "Pleural Effusion"   # which label Step 8 explains

# ── Training hyperparameters (sweep these) ───────────────────────────────
L1_LAMBDA = 1e-4
N_EPOCHS  = 100
LR        = 1e-3

# ── Hyperparameter sweep ──────────────────────────────────────────────────
RUN_SWEEP = False   # False → skip sweep cell entirely; True → sweep grids below instead of single-run values

# ── Concept filtering thresholds ─────────────────────────────────────────
MAX_LEN               = 50
SIM_TO_LABEL_THRESH   = 0.9
SIM_TO_CONCEPT_THRESH = 0.9
MIN_MAX_SIM           = 0.2   # measured: drops 0/95 concepts at current threshold — effectively inactive

# ── Infrastructure (rarely touch) ────────────────────────────────────────
DEVICE           = "mps"
BATCH_SIZE       = 32
CHECKPOINT_EVERY = 1000
TOP_K            = 5
IMG_ROOT         = Path("CheXpert-v1.0-small")
RAW_CSV_PATH     = "CheXpert-v1.0-small/CheXpert-v1.0-small/train.csv"

# ⚠  All output files share one name regardless of USE_SUBSAMPLE.
# If you flip the flag, delete img_emb_train.npy / img_emb_val.npy /
# img_emb_test.npy before rerunning — Step 4 will otherwise silently
# load stale embeddings from the other mode.

print(f"DEVICE: {DEVICE} | USE_SUBSAMPLE: {USE_SUBSAMPLE} | SEED: {SEED}")


DEVICE: mps | USE_SUBSAMPLE: True | SEED: 42


## Imports & setup

In [4]:
import os, json, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from sklearn.cluster import KMeans
from sklearn.metrics import roc_auc_score
from tqdm import tqdm
import open_clip
import mlflow

os.environ["HF_HUB_DISABLE_XET"] = "1"

mlflow.set_tracking_uri("sqlite:///mlflow.db")   # MLflow 3.x requires a DB backend, not plain ./mlruns
mlflow.set_experiment("cbm-hyperparam-tuning")

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.backends.mps.is_available():
        torch.mps.manual_seed(seed)

def encode_text(strings, model, tokenizer, device, context_length=256):
    """Tokenize → encode → L2-normalise a list of strings through BiomedCLIP text tower."""
    tokens = tokenizer(strings, context_length=context_length)
    with torch.no_grad():
        emb = model.encode_text(tokens.to(device))
    return emb / emb.norm(dim=-1, keepdim=True)

set_seed(SEED)
print("Imports done, seed set.")


Imports done, seed set.


In [5]:
df = pd.read_csv(RAW_CSV_PATH)
print(f"Raw CSV: {df.shape}")

# Drop uncertain labels (-1)
df = df[~(df[df.columns[5:]] == -1).any(axis="columns")]
# Fill missing labels with 0 (not mentioned = absent)
df[df.columns[5:]] = df[df.columns[5:]].fillna(0)
# Frontal only
df = df[df["Frontal/Lateral"] == "Frontal"].copy()

# Single source of truth — LABEL_COLS never redefined below
LABEL_COLS = list(df.columns[5:])

print(f"After filtering: {df.shape}")
print(f"Labels ({len(LABEL_COLS)}): {LABEL_COLS}")


Raw CSV: (223414, 19)
After filtering: (118286, 19)
Labels (14): ['No Finding', 'Enlarged Cardiomediastinum', 'Cardiomegaly', 'Lung Opacity', 'Lung Lesion', 'Edema', 'Consolidation', 'Pneumonia', 'Atelectasis', 'Pneumothorax', 'Pleural Effusion', 'Pleural Other', 'Fracture', 'Support Devices']


In [6]:
# Named biomed_model — cannot be clobbered by nn.Linear in Step 6
biomed_model, preprocess = open_clip.create_model_from_pretrained(
    "hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224"
)
tokenizer = open_clip.get_tokenizer(
    "hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224"
)
biomed_model = biomed_model.to(DEVICE).eval()
print(f"BiomedCLIP loaded and on {DEVICE}")


BiomedCLIP loaded and on mps


## Step 1 — Concept vocabulary

Reads `raw_concepts.json`, filters by length, similarity to class labels, and near-duplicate concepts.  
Saves `concepts_stage1.json`. Always recomputes.

In [7]:
raw = json.load(open("../data/raw_concepts.json"))

# Flatten + exact-dedup (case-insensitive)
seen, candidates = set(), []
for pathology, concept_list in raw.items():
    for c in concept_list:
        key = c.strip().lower()
        if key not in seen:
            seen.add(key)
            candidates.append(c.strip())
print(f"Flattened + deduped: {len(candidates)}")

# Length filter
candidates = [c for c in candidates if len(c) <= MAX_LEN]
print(f"After length filter (max {MAX_LEN} chars): {len(candidates)}")

# Encode labels and candidates through shared encode_text()
label_emb   = encode_text(LABEL_COLS,  biomed_model, tokenizer, DEVICE)
concept_emb = encode_text(candidates,  biomed_model, tokenizer, DEVICE)

# Drop concepts too similar to any class label
max_sim_to_label = (concept_emb @ label_emb.T).max(dim=1).values
keep = max_sim_to_label <= SIM_TO_LABEL_THRESH
candidates  = [c for c, k in zip(candidates, keep.tolist()) if k]
concept_emb = concept_emb[keep]
print(f"After label-similarity filter (thresh {SIM_TO_LABEL_THRESH}): {len(candidates)}")

# Greedy concept-concept dedup — keep first occurrence
sim = concept_emb @ concept_emb.T
dropped, final_concepts = set(), []
for i in range(len(candidates)):
    if i in dropped:
        continue
    final_concepts.append(candidates[i])
    for j in range(i + 1, len(candidates)):
        if j not in dropped and sim[i, j].item() > SIM_TO_CONCEPT_THRESH:
            dropped.add(j)
print(f"After concept-concept dedup (thresh {SIM_TO_CONCEPT_THRESH}): {len(final_concepts)}")

json.dump(final_concepts, open("concepts_stage1.json", "w"), indent=2)
print("Saved concepts_stage1.json")


Flattened + deduped: 107
After length filter (max 50 chars): 107


After label-similarity filter (thresh 0.9): 105


After concept-concept dedup (thresh 0.9): 95
Saved concepts_stage1.json


## Step 2 — Patient-level split

Splits on unique patient IDs (not rows) to prevent data leakage.  
Saves `train.csv`, `val.csv`, `test.csv`. Always recomputes.

In [8]:
df1 = df.copy()
df1["patient_id"] = df1["Path"].str.extract(r"(patient\d+)")
n_patients = df1["patient_id"].nunique()
print(f"Total: {len(df1)} rows, {n_patients} patients")

if USE_SUBSAMPLE:
    patient_labels   = df1.groupby("patient_id")[LABEL_COLS].max()
    label_prevalence = patient_labels.sum().sort_values()

    def rarest_label(row):
        positives = [l for l in LABEL_COLS if row[l] == 1]
        return min(positives, key=lambda l: label_prevalence[l]) if positives else "none"

    patient_labels["strat_key"] = patient_labels.apply(rarest_label, axis=1)
    rows_per_patient  = len(df1) / n_patients
    target_patients   = int(TARGET_ROWS / rows_per_patient)

    sampled = (
        patient_labels
        .groupby("strat_key", group_keys=False)
        .apply(lambda g: g.sample(
            n=max(1, round(len(g) * target_patients / n_patients)),
            random_state=SEED
        ))
        .index
    )
    sampled = pd.Index(sampled).unique()
    df1 = df1[df1["patient_id"].isin(sampled)].copy()
    print(f"Subsampled to {len(sampled)} patients, {len(df1)} rows")
else:
    print(f"Using full dataset: {n_patients} patients, {len(df1)} rows")

# Patient-level 80/10/10 split — deterministic via SEED
patients = df1["patient_id"].unique().astype(str)
rng = np.random.default_rng(SEED)
rng.shuffle(patients)
n       = len(patients)
n_train = int(0.8 * n)
n_val   = int(0.1 * n)

train_patients = set(patients[:n_train])
val_patients   = set(patients[n_train : n_train + n_val])
test_patients  = set(patients[n_train + n_val :])

assert not (train_patients & val_patients),  "train/val overlap"
assert not (train_patients & test_patients), "train/test overlap"
assert not (val_patients   & test_patients), "val/test overlap"

train_df = df1[df1["patient_id"].isin(train_patients)].copy().reset_index(drop=True)
val_df   = df1[df1["patient_id"].isin(val_patients)].copy().reset_index(drop=True)
test_df  = df1[df1["patient_id"].isin(test_patients)].copy().reset_index(drop=True)

train_df.to_csv("train.csv", index=False)
val_df.to_csv("val.csv",     index=False)
test_df.to_csv("test.csv",   index=False)

print(f"Patients → train {len(train_patients)}, val {len(val_patients)}, test {len(test_patients)}")
print(f"Rows     → train {len(train_df)}, val {len(val_df)}, test {len(test_df)}")
print("No patient overlap confirmed ✓")


Total: 118286 rows, 49404 patients


Subsampled to 4175 patients, 9817 rows
Patients → train 3340, val 417, test 418
Rows     → train 7811, val 978, test 1028
No patient overlap confirmed ✓


/var/folders/1f/h4qhh2cx0c1fc6x1f1v148540000gn/T/ipykernel_2586/3820241788.py:21: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(


## Step 3 — Text embeddings

Encodes `concepts_stage1.json` through BiomedCLIP text tower. L2-normalised.  
Saves `text_emb.pt`. Always recomputes.

In [9]:
concepts = json.load(open("concepts_stage1.json"))
print(f"Loaded {len(concepts)} concepts")

text_emb = encode_text(concepts, biomed_model, tokenizer, DEVICE)

torch.save(text_emb, "text_emb.pt")
print(f"Saved text_emb.pt — shape {text_emb.shape}")

norms = text_emb.norm(dim=-1)
assert torch.allclose(norms, torch.ones_like(norms), atol=1e-5), "L2 norm check failed"
print(f"Norm check passed — sample norms: {norms[:5].tolist()}")


Loaded 95 concepts


Saved text_emb.pt — shape torch.Size([95, 512])
Norm check passed — sample norms: [1.0, 1.0, 0.9999999403953552, 0.9999999403953552, 1.0]


## Step 4 — Image embeddings  *(the one cached step)*

Encodes every image through BiomedCLIP image tower. Checkpoints every `CHECKPOINT_EVERY` images  
so an interruption can resume rather than restart. L2-normalised. Saves `img_emb_{split}.npy`.

**Cache policy:** if the `.npy` file exists for a split, loads it. Delete the file to force re-encode.  
**Hard assert after load:** row count must match the CSV. If you flipped `USE_SUBSAMPLE` without  
deleting the cache files, this will catch it with a clear message.

In [10]:
def encode_split(df, split_name, model, preprocess, device):
    """
    Encodes all images in df through BiomedCLIP image tower, preserving CSV row order.
    Checkpoints every CHECKPOINT_EVERY images to avoid losing progress on interruption.
    Returns embedding array [len(df), 512].
    """
    out_path  = Path(f"img_emb_{split_name}.npy")
    ckpt_path = Path(f"img_emb_{split_name}_ckpt.npy")
    paths = df["Path"].tolist()
    n = len(paths)

    if ckpt_path.exists():
        done = np.load(ckpt_path)
        start_idx = len(done)
        print(f"[{split_name}] Resuming from checkpoint at {start_idx}/{n}")
    else:
        done = np.zeros((0, 512), dtype=np.float32)
        start_idx = 0
        print(f"[{split_name}] Starting fresh — {n} images")

    all_embs   = list(done)
    batch_imgs = []

    def flush():
        if not batch_imgs:
            return
        t = torch.stack(batch_imgs).to(device)
        with torch.no_grad():
            emb = model.encode_image(t)
            emb = emb / emb.norm(dim=-1, keepdim=True)
        all_embs.extend(emb.cpu().numpy())
        batch_imgs.clear()

    for i in tqdm(range(start_idx, n), desc=split_name):
        img = Image.open(IMG_ROOT / paths[i]).convert("RGB")
        batch_imgs.append(preprocess(img))
        if len(batch_imgs) == BATCH_SIZE:
            flush()
        if (i + 1) % CHECKPOINT_EVERY == 0:
            np.save(ckpt_path, np.array(all_embs, dtype=np.float32))

    flush()
    final = np.array(all_embs, dtype=np.float32)
    np.save(out_path, final)
    if ckpt_path.exists():
        ckpt_path.unlink()
    print(f"[{split_name}] Done — shape {final.shape}")
    return final


# Per-split cache check — only re-encodes missing splits
img_embs = {}
for split_name, split_df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    out_path = Path(f"img_emb_{split_name}.npy")
    if out_path.exists():
        img_embs[split_name] = np.load(out_path)
        print(f"[{split_name}] Loaded from cache — shape {img_embs[split_name].shape}")
    else:
        img_embs[split_name] = encode_split(split_df, split_name, biomed_model, preprocess, DEVICE)

# Hard alignment asserts — the USE_SUBSAMPLE flag tripwire
assert img_embs["train"].shape[0] == len(train_df), (
    f"train mismatch ({img_embs['train'].shape[0]} vs {len(train_df)}) — "
    "delete img_emb_train.npy and rerun (likely stale after flipping USE_SUBSAMPLE)"
)
assert img_embs["val"].shape[0] == len(val_df), (
    f"val mismatch — delete img_emb_val.npy and rerun"
)
assert img_embs["test"].shape[0] == len(test_df), (
    f"test mismatch — delete img_emb_test.npy and rerun"
)
print("Alignment confirmed ✓")


[train] Loaded from cache — shape (7811, 512)
[val] Loaded from cache — shape (978, 512)
[test] Loaded from cache — shape (1028, 512)
Alignment confirmed ✓


## Step 5 — Concept score matrix

Computes cosine similarity between every image and every concept.  
Applies filter 3 (drop if max train similarity < `MIN_MAX_SIM`).  
Binarizes with 2-means KMeans — fit on train, applied to val/test (no leakage).  
Runs sanity checks. Always recomputes.

In [11]:
concepts = json.load(open("concepts_stage1.json"))
text_emb = torch.load("text_emb.pt").to(DEVICE)

def compute_scores(img_emb_arr):
    img_t = torch.from_numpy(img_emb_arr.astype(np.float32)).to(DEVICE)
    with torch.no_grad():
        scores = img_t @ text_emb.T
    return scores.cpu().numpy()

scores = {split: compute_scores(img_embs[split]) for split in ("train", "val", "test")}
print("Raw score shapes:", {k: v.shape for k, v in scores.items()})

# Filter 3: drop concepts with max train similarity < MIN_MAX_SIM
max_sim   = scores["train"].max(axis=0)
keep_mask = max_sim >= MIN_MAX_SIM
n_dropped = int((~keep_mask).sum())
dropped_names = [c for c, k in zip(concepts, keep_mask) if not k]
print(f"Filter 3: dropped {n_dropped} concepts (max_sim < {MIN_MAX_SIM})")
if dropped_names:
    print(f"  Dropped: {dropped_names}")

concepts_final = [c for c, k in zip(concepts, keep_mask) if k]
for split in scores:
    scores[split] = scores[split][:, keep_mask]

json.dump(concepts_final, open("../data/concepts_final.json", "w"), indent=2)
for split in scores:
    np.save(f"concept_scores_{split}.npy", scores[split])
print(f"Saved concepts_final.json ({len(concepts_final)} concepts) and raw concept scores")


def binarize(score_arr, kmeans_models=None, fit=False):
    """
    Binarize each concept column with 2-means KMeans.
    fit=True: fit on this data, return (binary_matrix, models).
    fit=False: apply pre-fitted models from train split.
    Higher-centroid cluster → 1, lower → 0.
    """
    binary = np.zeros_like(score_arr, dtype=np.int8)
    models = [] if fit else kmeans_models
    for j in range(score_arr.shape[1]):
        col = score_arr[:, j].reshape(-1, 1)
        if fit:
            km = KMeans(n_clusters=2, random_state=SEED, n_init=10)
            km.fit(col)
            models.append(km)
        else:
            km = kmeans_models[j]
        labels = km.predict(col)
        c0, c1 = km.cluster_centers_[0][0], km.cluster_centers_[1][0]
        high_label = 0 if c0 > c1 else 1
        binary[:, j] = (labels == high_label).astype(np.int8)
    return (binary, models) if fit else binary


def minmax_scale_per_image(score_arr):
    """
    Per-image min-max scaling across concepts (Kolek et al. 2023, arXiv:2312.11548,
    Eq. 22-24): for each image (row), rescale its own concept scores using that
    image's own min/max across all concepts. Every image's top-matching concept
    maps to 1.0, weakest-matching maps to 0.0. No fit/apply split needed — computed
    independently per row, so no leakage risk.
    """
    row_min = score_arr.min(axis=1, keepdims=True)
    row_max = score_arr.max(axis=1, keepdims=True)
    denom   = np.clip(row_max - row_min, a_min=1e-8, a_max=None)
    return ((score_arr - row_min) / denom).astype(np.float32)


def minmax_scale_per_concept(score_arr, stats=None, fit=False):
    """
    Per-concept min-max scaling across images: for each concept (column), rescale
    using min/max computed from train only (fit=True), applied to val/test
    (fit=False) to avoid leakage. Clips to [0,1] so val/test extremes don't
    extrapolate past the train-derived range.
    """
    if fit:
        col_min, col_max = score_arr.min(axis=0), score_arr.max(axis=0)
        stats = (col_min, col_max)
    else:
        col_min, col_max = stats
    denom  = np.clip(col_max - col_min, a_min=1e-8, a_max=None)
    scaled = np.clip((score_arr - col_min) / denom, 0.0, 1.0).astype(np.float32)
    return (scaled, stats) if fit else scaled


concept_variants = {}

# Variant A: original K-means binarization
cm_train, kmeans_models = binarize(scores["train"], fit=True)
concept_variants["binarized"] = {
    "train": cm_train,
    "val":   binarize(scores["val"],  kmeans_models=kmeans_models),
    "test":  binarize(scores["test"], kmeans_models=kmeans_models),
}

# Variant B: per-image min-max
concept_variants["minmax_per_image"] = {
    split: minmax_scale_per_image(scores[split]) for split in ("train", "val", "test")
}

# Variant C: per-concept min-max
cm_train_c, concept_scale_stats = minmax_scale_per_concept(scores["train"], fit=True)
concept_variants["minmax_per_concept"] = {
    "train": cm_train_c,
    "val":   minmax_scale_per_concept(scores["val"],  stats=concept_scale_stats),
    "test":  minmax_scale_per_concept(scores["test"], stats=concept_scale_stats),
}

for variant_name, mats in concept_variants.items():
    for split, mat in mats.items():
        np.save(f"concept_matrix_{variant_name}_{split}.npy", mat)
print(f"Saved concept matrix variants: {list(concept_variants.keys())}")

# ── Sanity checks ─────────────────────────────────────────────────────────
print("\nCentroid separation per concept, binarized variant (flag if < 0.05):")
low_sep = [
    (concepts_final[j], abs(km.cluster_centers_[0][0] - km.cluster_centers_[1][0]))
    for j, km in enumerate(kmeans_models)
    if abs(km.cluster_centers_[0][0] - km.cluster_centers_[1][0]) < 0.05
]
if low_sep:
    print(f"  {len(low_sep)} concepts with low separation:")
    for name, sep in low_sep:
        print(f"    '{name}': {sep:.4f}")
else:
    print("  All concepts have healthy separation ✓")

act_rate = concept_variants["binarized"]["train"].mean(axis=0)
print(f"\nActivation rate (binarized) — min {act_rate.min():.3f}, max {act_rate.max():.3f}, mean {act_rate.mean():.3f}")
degenerate = [(concepts_final[j], act_rate[j])
              for j in range(len(concepts_final))
              if act_rate[j] < 0.01 or act_rate[j] > 0.99]
if degenerate:
    print(f"  Degenerate concepts (nearly all 0 or all 1): {degenerate}")
else:
    print("  No degenerate concepts ✓")

print("\nContinuous-scaling dynamic-range check (raw scores before scaling):")
for variant_name in ("minmax_per_image", "minmax_per_concept"):
    axis = 1 if variant_name == "minmax_per_image" else 0
    ranges = scores["train"].max(axis=axis) - scores["train"].min(axis=axis)
    n_flat = int((ranges < 0.05).sum())
    unit = "images" if axis == 1 else "concepts"
    print(f"  [{variant_name}] {n_flat}/{len(ranges)} {unit} have near-flat score range (<0.05) before scaling")
    mat = concept_variants[variant_name]["train"]
    print(f"  [{variant_name}] scaled score mean — min {mat.mean(axis=axis).min():.3f}, "
          f"max {mat.mean(axis=axis).max():.3f}, overall mean {mat.mean():.3f}")


Raw score shapes: {'train': (7811, 95), 'val': (978, 95), 'test': (1028, 95)}
Filter 3: dropped 0 concepts (max_sim < 0.2)
Saved concepts_final.json (95 concepts) and raw concept scores


Saved concept matrix variants: ['binarized', 'minmax_per_image', 'minmax_per_concept']

Centroid separation per concept, binarized variant (flag if < 0.05):
  32 concepts with low separation:
    'clear lung fields': 0.0422
    'symmetric lung inflation': 0.0427
    'widened vascular pedicle': 0.0494
    'loss of mediastinal margins': 0.0464
    'hazy lung opacity': 0.0449
    'increased lung density': 0.0408
    'airspace opacification': 0.0438
    'patchy parenchymal opacity': 0.0486
    'diffuse ground glass opacity': 0.0479
    'focal consolidative opacity': 0.0476
    'irregular lung mass': 0.0475
    'interstitial lung markings': 0.0422
    'kerley b lines': 0.0495
    'peribronchial cuffing': 0.0445
    'vascular redistribution': 0.0416
    'fluid in fissures': 0.0451
    'segmental consolidation': 0.0479
    'unilateral lung infiltrate': 0.0494
    'volume loss in lung': 0.0442
    'shifted fissure': 0.0416
    'platelike atelectasis': 0.0489
    'collapsed lung segment': 0.034

## Step 6 — Train the linear probe

One shared `train_linear_probe()` function for both the CBM head and the baseline.  
CBM head: 95-concept matrix → 14 labels, with L1 regularisation for sparsity.  
Baseline: raw 512-d CLIP embeddings → 14 labels, no L1 (dense probe for comparison).  
Always recomputes.

In [12]:
def train_linear_probe(X_train, y_train_t, X_val, y_val_t, n_labels,
                       ckpt_path, l1_lambda=L1_LAMBDA,
                       n_epochs=N_EPOCHS, lr=LR, seed=SEED,
                       loss_fn=None, optimizer_cls=None, patience=None):
    """
    Single nn.Linear(n_features, n_labels) trained with BCEWithLogitsLoss.
    Saves best-val-AUROC checkpoint to ckpt_path.
    Logs params + per-epoch train/val loss and val AUROC to MLflow.
    patience=None (default): trains for exactly n_epochs, no early stopping.
    patience=k: stops once val AUROC hasn't improved for k consecutive epochs.
    Returns (probe, best_val_auroc).
    """
    set_seed(seed)
    loss_fn       = loss_fn or nn.BCEWithLogitsLoss
    optimizer_cls = optimizer_cls or torch.optim.Adam
    probe         = nn.Linear(X_train.shape[1], n_labels).to(DEVICE)
    criterion     = loss_fn()
    optimizer     = optimizer_cls(probe.parameters(), lr=lr)

    best_val_auroc       = 0.0
    epochs_since_improve = 0
    y_val_np             = y_val_t.cpu().numpy()

    with mlflow.start_run(run_name=Path(ckpt_path).stem):
        mlflow.log_params({
            "lr": lr, "l1_lambda": l1_lambda, "n_epochs": n_epochs,
            "loss_fn": loss_fn.__name__, "optimizer": optimizer_cls.__name__,
            "patience": patience,
        })

        for epoch in range(n_epochs):
            probe.train()
            optimizer.zero_grad()
            logits = probe(X_train)
            loss   = criterion(logits, y_train_t)
            if l1_lambda > 0:
                l1   = sum(p.abs().sum() for p in probe.parameters())
                loss = loss + l1_lambda * l1
            loss.backward()
            optimizer.step()

            probe.eval()
            with torch.no_grad():
                val_logits = probe(X_val)
                val_loss   = criterion(val_logits, y_val_t).item()
                val_probs  = torch.sigmoid(val_logits).cpu().numpy()

            aurocs = [
                roc_auc_score(y_val_np[:, i], val_probs[:, i])
                for i in range(n_labels)
                if len(np.unique(y_val_np[:, i])) > 1
            ]
            macro = np.mean(aurocs)

            mlflow.log_metrics(
                {"train_loss": loss.item(), "val_loss": val_loss, "val_auroc": macro},
                step=epoch,
            )

            if macro > best_val_auroc:
                best_val_auroc       = macro
                epochs_since_improve = 0
                torch.save(probe.state_dict(), ckpt_path)
            else:
                epochs_since_improve += 1

            if epoch % 10 == 0:
                print(f"  epoch {epoch:3d}  loss {loss.item():.4f}  val AUROC {macro:.4f}")

            if patience is not None and epochs_since_improve >= patience:
                print(f"  Early stop at epoch {epoch} (no improvement for {patience} epochs)")
                break

        mlflow.log_metric("best_val_auroc", best_val_auroc)

    probe.load_state_dict(torch.load(ckpt_path))
    probe.eval()
    return probe, best_val_auroc


In [13]:
# Load labels from the CSVs saved in Step 2
# Row order matches img_emb_*.npy and concept_matrix_*.npy exactly
train_df = pd.read_csv("train.csv")
val_df   = pd.read_csv("val.csv")
test_df  = pd.read_csv("test.csv")

y_train = train_df[LABEL_COLS].values
y_val   = val_df[LABEL_COLS].values
y_test  = test_df[LABEL_COLS].values

# Baseline inputs: raw 512-d CLIP embeddings (dense probe, no bottleneck)
Xb_train = torch.tensor(img_embs["train"], dtype=torch.float32).to(DEVICE)
Xb_val   = torch.tensor(img_embs["val"],   dtype=torch.float32).to(DEVICE)
Xb_test  = torch.tensor(img_embs["test"],  dtype=torch.float32).to(DEVICE)

# Labels as float tensors on device
y_train_t = torch.tensor(y_train, dtype=torch.float32).to(DEVICE)
y_val_t   = torch.tensor(y_val,   dtype=torch.float32).to(DEVICE)
y_test_t  = torch.tensor(y_test,  dtype=torch.float32).to(DEVICE)

n_labels = len(LABEL_COLS)
print(f"Concept variants: {list(concept_variants.keys())}, each {concept_variants['binarized']['train'].shape}")
print(f"Baseline input: Xb_train {Xb_train.shape}")
print(f"Labels:         y_train {y_train_t.shape}")


Concept variants: ['binarized', 'minmax_per_image', 'minmax_per_concept'], each (7811, 95)
Baseline input: Xb_train torch.Size([7811, 512])
Labels:         y_train torch.Size([7811, 14])


In [14]:
VARIANT_TRAIN_CONFIG = {
    "binarized":          {"n_epochs": 100,  "patience": None},
    "minmax_per_image":   {"n_epochs": 2000, "patience": 100},
    "minmax_per_concept": {"n_epochs": 2000, "patience": 100},
}

cbm_results = {}
for variant_name, mats in concept_variants.items():
    Xv_train = torch.tensor(mats["train"], dtype=torch.float32).to(DEVICE)
    Xv_val   = torch.tensor(mats["val"],   dtype=torch.float32).to(DEVICE)
    Xv_test  = torch.tensor(mats["test"],  dtype=torch.float32).to(DEVICE)

    cfg = VARIANT_TRAIN_CONFIG[variant_name]
    print(f"\nTraining CBM probe — variant: {variant_name} "
          f"(n_epochs={cfg['n_epochs']}, patience={cfg['patience']})")
    probe, best_val = train_linear_probe(
        Xv_train, y_train_t, Xv_val, y_val_t,
        n_labels   = n_labels,
        ckpt_path  = f"cbm_probe_{variant_name}.pt",
        l1_lambda  = L1_LAMBDA,
        n_epochs   = cfg["n_epochs"],
        patience   = cfg["patience"],
    )
    cbm_results[variant_name] = {"probe": probe, "X_test": Xv_test, "best_val_auroc": best_val}
    print(f"[{variant_name}] best val AUROC: {best_val:.4f}")

print("\nTraining baseline probe (raw CLIP embeddings, no concept bottleneck)...")
baseline_probe, baseline_best_val = train_linear_probe(
    Xb_train, y_train_t, Xb_val, y_val_t,
    n_labels   = n_labels,
    ckpt_path  = "baseline_probe_best.pt",
    l1_lambda  = 0.0,   # no sparsity on baseline — dense probe
)
print(f"Baseline best val AUROC: {baseline_best_val:.4f}")



Training CBM probe — variant: binarized (n_epochs=100, patience=None)


  epoch   0  loss 0.7105  val AUROC 0.4792


  epoch  10  loss 0.5774  val AUROC 0.5051


  epoch  20  loss 0.4957  val AUROC 0.5247


  epoch  30  loss 0.4496  val AUROC 0.5373


...

  epoch  60  loss 0.4861  val AUROC 0.6183


  epoch  70  loss 0.4675  val AUROC 0.6226


  epoch  80  loss 0.4518  val AUROC 0.6267


  epoch  90  loss 0.4385  val AUROC 0.6309


Baseline best val AUROC: 0.6345


## Step 6b — Hyperparameter sweep (optional)

Set `RUN_SWEEP = True` in Config to sweep learning rate, L1 lambda, loss, and optimizer for the CBM probe.  
Leave `RUN_SWEEP = False` (default) to skip — the single-run training above is unaffected either way.  
Saves `hyperparam_sweep_results.csv` ranked by validation AUROC, and one checkpoint per combo (`cbm_probe_<tag>.pt`).

In [15]:
from itertools import product

if not RUN_SWEEP:
    print("RUN_SWEEP is False — skipping sweep (using single-run LR/L1_LAMBDA from Config instead).")
else:
    SWEEP_VARIANT = "binarized"   # which concept_variants entry to sweep against — change to compare sweep results across variants
    mats = concept_variants[SWEEP_VARIANT]
    Xs_train = torch.tensor(mats["train"], dtype=torch.float32).to(DEVICE)
    Xs_val   = torch.tensor(mats["val"],   dtype=torch.float32).to(DEVICE)
    print(f"Sweeping against concept variant: {SWEEP_VARIANT}")

    LR_GRID    = [1e-4, 1e-3, 1e-2]
    L1_GRID    = [0.0, 1e-4]
    LOSS_GRID  = {"bce": nn.BCEWithLogitsLoss}   # add more losses here, e.g. "focal": FocalLoss
    OPTIM_GRID = {"adam": torch.optim.Adam, "sgd": torch.optim.SGD}  # add more optimizers here

    results = []
    for lr, l1, (loss_name, loss_fn), (optim_name, optim_cls) in product(
        LR_GRID, L1_GRID, LOSS_GRID.items(), OPTIM_GRID.items()
    ):
        tag = f"{SWEEP_VARIANT}_lr{lr}_l1{l1}_{loss_name}_{optim_name}"
        print(f"--- {tag} ---")
        probe, val_auroc = train_linear_probe(
            Xs_train, y_train_t, Xs_val, y_val_t,
            n_labels=n_labels,
            ckpt_path=f"cbm_probe_{tag}.pt",   # unique checkpoint per combo — no overwriting
            l1_lambda=l1, lr=lr, loss_fn=loss_fn, optimizer_cls=optim_cls,
        )
        results.append({"variant": SWEEP_VARIANT, "lr": lr, "l1_lambda": l1,
                         "loss": loss_name, "optimizer": optim_name, "val_auroc": val_auroc})

    results_df = pd.DataFrame(results).sort_values("val_auroc", ascending=False)
    results_df.to_csv("hyperparam_sweep_results.csv", index=False)
    results_df


RUN_SWEEP is False — skipping sweep (using single-run LR/L1_LAMBDA from Config instead).


## Step 7 — Evaluate

Per-label AUROC on the test set, macro averages for CBM and baseline,  
accuracy cost of the bottleneck, and a faithfulness table.

In [16]:
baseline_probe.eval()
with torch.no_grad():
    test_probs_base = torch.sigmoid(baseline_probe(Xb_test)).cpu().numpy()

y_test_np = y_test_t.cpu().numpy()

# Baseline macro
base_aurocs = [
    roc_auc_score(y_test_np[:, i], test_probs_base[:, i])
    for i in range(n_labels)
    if len(np.unique(y_test_np[:, i])) > 1
]
macro_base = np.mean(base_aurocs)

# Per-variant CBM test AUROC
comparison_rows = []
for variant_name, res in cbm_results.items():
    res["probe"].eval()
    with torch.no_grad():
        probs = torch.sigmoid(res["probe"](res["X_test"])).cpu().numpy()
    aurocs = [
        roc_auc_score(y_test_np[:, i], probs[:, i])
        for i in range(n_labels) if len(np.unique(y_test_np[:, i])) > 1
    ]
    comparison_rows.append({
        "variant": variant_name,
        "test_macro_auroc": np.mean(aurocs),
        "best_val_auroc": res["best_val_auroc"],
    })

comparison_rows.append({
    "variant": "baseline_no_bottleneck",
    "test_macro_auroc": macro_base,
    "best_val_auroc": baseline_best_val,
})

comparison_df = pd.DataFrame(comparison_rows).sort_values("test_macro_auroc", ascending=False)
comparison_df.to_csv("../results/concept_scaling_comparison.csv", index=False)

# Select which variant the faithfulness table below and Step 8's explain() use —
# resolved here since the very next cell already needs to know which probe/weights to inspect.
EXPLAIN_VARIANT = comparison_df.iloc[0]["variant"]   # defaults to the best-performing CBM variant; override to inspect a specific one
if EXPLAIN_VARIANT == "baseline_no_bottleneck":
    EXPLAIN_VARIANT = comparison_df[comparison_df["variant"] != "baseline_no_bottleneck"].iloc[0]["variant"]
    print(f"Baseline won on AUROC, but it has no concept bottleneck to explain — using best CBM variant instead: {EXPLAIN_VARIANT}")

cbm_probe = cbm_results[EXPLAIN_VARIANT]["probe"]
cbm_probe.eval()
with torch.no_grad():
    test_probs_cbm = torch.sigmoid(cbm_probe(cbm_results[EXPLAIN_VARIANT]["X_test"])).cpu().numpy()

print(f"Accuracy cost of bottleneck ({EXPLAIN_VARIANT} vs. baseline): "
      f"{macro_base - comparison_df.set_index('variant').loc[EXPLAIN_VARIANT, 'test_macro_auroc']:.4f}")
comparison_df


Accuracy cost of bottleneck (minmax_per_concept vs. baseline): -0.0533


,variant,test_macro_auroc,best_val_auroc
2,minmax_per_concept,0.680889,0.682280
1,minmax_per_image,0.679542,0.705393
3,baseline_no_bottleneck,0.627635,0.634467
0,binarized,0.603282,0.584177


In [17]:
concept_matrix_train_explain = concept_variants[EXPLAIN_VARIANT]["train"]
weights = cbm_probe.weight.detach().cpu().numpy()   # [n_labels, n_concepts]

print(f"Faithfulness table — variant: {EXPLAIN_VARIANT}\n")
print(f"{'Label':<28} {'Top concept':<32} {'weight':>7} {'pos_mean':>9} {'neg_mean':>9} {'ok?':>5}")
for i, name in enumerate(LABEL_COLS):
    top_j = np.argmax(np.abs(weights[i]))
    w     = weights[i, top_j]
    pos_mask = y_train[:, i] == 1
    neg_mask = y_train[:, i] == 0
    pos_r = concept_matrix_train_explain[pos_mask, top_j].mean() if pos_mask.sum() > 0 else float("nan")
    neg_r = concept_matrix_train_explain[neg_mask, top_j].mean() if neg_mask.sum() > 0 else float("nan")
    # Faithful if weight direction matches pos/neg difference
    faithful = "✓" if (w > 0 and pos_r > neg_r) or (w < 0 and pos_r < neg_r) else "✗"
    print(f"{name:<28} {concepts_final[top_j]:<32} {w:7.3f} {pos_r:9.3f} {neg_r:9.3f} {faithful:>5}")


Faithfulness table — variant: minmax_per_concept

Label                        Top concept                       weight  pos_mean  neg_mean   ok?
No Finding                   mediastinal shift from large effusion  -0.684     0.397     0.540     ✓
Enlarged Cardiomediastinum   widened mediastinal silhouette    -0.208     0.682     0.673     ✗
Cardiomegaly                 biventricular enlargement          0.823     0.547     0.390     ✓
Lung Opacity                 diffuse ground glass opacity       0.830     0.616     0.592     ✓
Lung Lesion                  endotracheal tube                 -0.311     0.490     0.587     ✓
Edema                        lucent area without vessels       -0.777     0.428     0.544     ✓
Consolidation                unremarkable chest radiograph     -0.335     0.450     0.554     ✓
Pneumonia                    nasogastric tube                  -0.256     0.484     0.594     ✓
Atelectasis                  calcified granuloma               -0.452     0.433  

## Step 8 — Interpretability

For `TARGET_LABEL`, shows the top-K driving concepts for high-confidence correct cases,  
then demonstrates a concept intervention (flip one concept, rerun the probe).

In [18]:
concept_matrix_test_explain = concept_variants[EXPLAIN_VARIANT]["test"]

label_idx     = LABEL_COLS.index(TARGET_LABEL)
label_weights = cbm_probe.weight.detach().cpu().numpy()[label_idx]   # [n_concepts]

pred_scores = test_probs_cbm[:, label_idx]
true_labels = y_test_np[:, label_idx]

high_conf_pos = np.where((pred_scores > 0.6) & (true_labels == 1))[0]
high_conf_neg = np.where((pred_scores < 0.4) & (true_labels == 0))[0]

print(f"Explaining variant: {EXPLAIN_VARIANT}")
print(f"High-confidence correct positives: {len(high_conf_pos)}")
print(f"High-confidence correct negatives: {len(high_conf_neg)}")


def explain(row_idx):
    """
    Print the top-K driving concepts for one test image, then show
    what happens when the most influential concept is ablated to 0.0
    (the weakest observed value for this concept/variant).
    """
    cm_row        = concept_matrix_test_explain[row_idx]
    contributions = label_weights * cm_row
    top_indices   = np.argsort(np.abs(contributions))[::-1][:TOP_K]

    pred = pred_scores[row_idx]
    true = int(true_labels[row_idx])
    path = test_df.iloc[row_idx]["Path"]

    print(f"\nImage: {path}")
    print(f"True: {true}  |  Score: {pred:.3f}")
    print(f"{'Concept':<32} {'score':>7} {'weight':>8} {'contribution':>14}")
    for j in top_indices:
        print(f"{concepts_final[j]:<32} {cm_row[j]:>7.3f} "
              f"{label_weights[j]:>8.3f} {contributions[j]:>14.4f}")

    # Intervention: ablate the top concept to 0.0 and rerun the probe
    top_j      = top_indices[0]
    orig_val   = float(cm_row[top_j])
    intervened = cm_row.copy().astype(np.float32)
    intervened[top_j] = 0.0   # ablate to the weakest observed value for this concept/variant
    with torch.no_grad():
        new_score = torch.sigmoid(
            cbm_probe(torch.tensor(intervened, dtype=torch.float32).to(DEVICE))
        )[label_idx].item()
    print(f"\nIntervention: ablate '{concepts_final[top_j]}' "
          f"{orig_val:.3f} → 0.000")
    print(f"Score: {pred:.3f} → {new_score:.3f}")


Explaining variant: minmax_per_concept
High-confidence correct positives: 207
High-confidence correct negatives: 406


In [19]:
print("=" * 65)
print(f"Explanations for {TARGET_LABEL}")
print("=" * 65)
for idx in high_conf_pos[:3]:
    explain(idx)
for idx in high_conf_neg[:2]:
    explain(idx)


Explanations for Pleural Effusion

Image: CheXpert-v1.0-small/train/patient00183/study3/view1_frontal.jpg
True: 1  |  Score: 0.693
Concept                            score   weight   contribution
mediastinal shift from large effusion   0.738    0.679         0.5014
boot shaped heart                  0.862    0.377         0.3252
widened vascular pedicle           0.770   -0.410        -0.3153
loculated pleural fluid            0.428    0.733         0.3139
shifted fissure                    0.479   -0.577        -0.2762

Intervention: ablate 'mediastinal shift from large effusion' 0.738 → 0.000
Score: 0.693 → 0.577

Image: CheXpert-v1.0-small/train/patient01187/study7/view1_frontal.jpg
True: 1  |  Score: 0.743
Concept                            score   weight   contribution
pleural thickening                 0.506    1.018         0.5151
patchy parenchymal opacity         0.583   -0.841        -0.4907
loculated pleural fluid            0.614    0.733         0.4499
mediastinal shift fr